<a href="https://colab.research.google.com/github/Ken89MathCompSci/UKDALE-NILM/blob/master/In-Depth-troubleshooting-UKDALE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content/UKDALE-NILM

/content/UKDALE-NILM


In [ ]:
%pwd

'/content/UKDALE-NILM'

In [3]:
!python fixed_pinn_lnn_ukdale.py --dataset-dir medium_dataset

Loading UKDALE CSV data ...
  train : 432000 rows  2014-11-01 -> 2014-11-30
  val   : 100800 rows  2014-12-01 -> 2014-12-07
  test  : 100800 rows  2014-08-24 -> 2014-08-30
  Columns: ['aggregate', 'dishwasher', 'fridge', 'microwave', 'washing_machine']

Device: cuda  |  WIN=299  hidden=64  dt=0.1
Fixes applied: sigmoid heads, Seq2Seq, BiLNN, physics@all-t, Z-score, 8-ch features, BCEWithLogits, adaptive thresholds

  Eval thresholds (W):        [used for metrics]
  Appliance           Train      Val     Test
  dishwasher           21.0     21.0     76.0
  fridge               10.0     10.0     10.0
  microwave            21.0     21.0     68.0
  washing_machine      23.0     23.0     34.0
  Event BCE thresholds (W):   [used for training targets]
  dishwasher           21.0
  fridge              104.0
  microwave            21.0
  washing_machine      23.0

Creating sequences ...
  Train : (43171, 299, 8) -> (43171, 299, 4)  (12,908,129 predictions)
  Val   : (10051, 299, 8) -> (10051, 

In [10]:
!python analyse_representations.py --model-dir models/fixed_pinn_lnn_20260517_081917

Device: cuda
Loaded config from models/fixed_pinn_lnn_20260517_081917/fixed_pinn_lnn_results.json

Loading and preprocessing data ...
  X_tr=(1411, 299, 8)  X_va=(1411, 299, 8)  X_te=(48, 299, 8)
Loaded model from models/fixed_pinn_lnn_20260517_081917/best_model.pt

  train done
  val done
  test done

[1] Weight Analysis
   CNN layers: ['conv0', 'conv1', 'conv2', 'conv3']
     conv0  eff.rank=19.26  top-5σ=['3.783', '3.496', '2.895', '2.829', '2.551']
     conv1  eff.rank=38.04  top-5σ=['4.753', '4.250', '3.933', '3.789', '3.476']
     conv2  eff.rank=41.12  top-5σ=['3.405', '2.937', '2.807', '2.739', '2.709']
     conv3  eff.rank=41.25  top-5σ=['3.335', '2.958', '2.804', '2.602', '2.378']
   Forward LNN: spectral_radius=3.3080  τ=[1.269, 1.743] (mean=1.424)
   Backward LNN: spectral_radius=2.4620  τ=[1.248, 1.732] (mean=1.408)
   Head weight norms:
     dishwasher          power=1.1842  event=0.5849
     fridge              power=1.0097  event=0.5601
     microwave           power=1.

In [ ]:
!python preprocess_hf.py


Processing split: train  (House 1, 2014-11-01)
  Loading channel_1  ->  aggregate
Traceback (most recent call last):
  File "/content/UKDALE-NILM/preprocess_hf.py", line 153, in <module>
    preprocess_split(split)
  File "/content/UKDALE-NILM/preprocess_hf.py", line 123, in preprocess_split
    raw = load_channel(house_dir, ch_num)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/UKDALE-NILM/preprocess_hf.py", line 79, in load_channel
    df   = pd.read_csv(path, sep=" ", header=None, names=["ts", "power"],
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

In [ ]:
!python test_pinn_lnn_ukdale_specific_splits_minonoff.py --dataset-dir medium_dataset

Loading CSVs from 'medium_dataset/'...
  train       : (432000, 5)  2014-11-01 → 2014-11-30
  validation  : (100800, 5)  2014-12-01 → 2014-12-07
  test        : (100800, 5)  2014-08-24 → 2014-08-30
  Columns: ['aggregate', 'dishwasher', 'fridge', 'microwave', 'washing_machine']
Using device: cuda
λ_phys=0.01  ε=50.0 W  hidden=64  dt=0.1
Min ON  : {'dishwasher': 300, 'fridge': 10, 'microwave': 2, 'washing_machine': 300}
Min OFF : {'dishwasher': 300, 'fridge': 2, 'microwave': 5, 'washing_machine': 27}
Train: (86380, 100, 1) → (86380, 4)
Val:   (20140, 100, 1) → (20140, 4)
Test:  (20140, 100, 1) → (20140, 4)
Model parameters: 9,028
Starting PINN-LNN training (all appliances simultaneously)...
  Epoch   1/80  train=0.00320 (mse=0.00315 phys=6.29888)  val=0.00564 (mse=0.00385 phys=0.17926)  avgF1=0.2408  avgMAE=80.70  lr=1.00e-03
    dishwasher      F1=0.0000  P=0.0000  R=0.0000  MAE=61.11  TP=0  FP=0  FN=767
    fridge          F1=0.6183  P=0.4479  R=0.9982  MAE=46.31  TP=9012  FP=11110  F